# [v2] Static per-scan classification — GAAE + LogReg

**Task: static single-scan classification.** Each scan is one sample;
labels are the subject's eventual converter status. This is *not* longitudinal
and *not* early-detection — it tests whether a single scan's GAAE embedding
is predictive on its own.

v2 adds **subject-level rollup** of the scan-level AUC at the end of the
notebook. Quote both numbers in any final table.


In [ ]:
# === Papermill parameters (injected by run_experiment.py) ===
# Safe interactive defaults: None keeps the original Jupyter behaviour
# (interactive checkpoint/threshold prompts, JSON-config loading).
EXPERIMENT_ID = None
MODE = None
MODEL = None
DATASET = None
SEED = 42
GAAE_CHECKPOINT_PATH = None   # None -> interactive checkpoint picker
THRESHOLD_MODE = None         # None -> interactive prompt; else youden | best-f1 | fixed
FIXED_THRESHOLD = None        # required when THRESHOLD_MODE is fixed
WANDB_ENABLED = True          # W&B logging is on by default
OUTPUT_DIR = None             # defaults to outputs/<experiment-id>/ when run standalone
RESOLVED_CONFIG = None        # merged hyperparameter dict; overrides on-disk JSON when set
RUN_DIR = None                # set by the runner: where run_summary.json / artifacts go
RUN_NAME = None               # set by the runner: the W&B run name


# GAAE Latent Space → Logistic Regression Classifier

Encoder-frozen GAAE → mean-pool graph embeddings → logistic regression head for converter vs stable-MCI classification.

Follows the same 5-fold stratified subject-level CV structure as the Cost-Weighted GEC notebook.

In [ ]:
import sys
from pathlib import Path
repo_root = Path('/mnt/e/fyassine/ad-early-detection')
model_root = Path('/mnt/e/fyassine/ad-early-detection/CLASSIFIER')
if str(model_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
    sys.path.insert(0, str(model_root))

In [ ]:
# v2 reproducibility seeding — must run before datasets, samplers, or models.
from CLASSIFIER.common.seeding import (
    set_seed, make_rng, make_torch_generator, seed_worker,
)
SEED = 42
set_seed(SEED)
rng = make_rng(SEED)
torch_gen = make_torch_generator(SEED)


In [ ]:
import json, os, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix,
    f1_score, classification_report
)
from torch_geometric.loader import DataLoader
from torch_geometric.utils import unbatch
from model.GAAE.models import GraphAttentionAutoencoderConditioned
from model.GAAE.dataset import GraphDatasetInMemoryFiltered
from model.GAAE.utils import knn_binary_adjacency_matrix_no_diag
from common.provenance import (
    region_from_data_root, make_run_dir, snapshot_source,
    write_run_summary, patch_run_summary, capture_env, capture_git_provenance,
)
warnings.filterwarnings('ignore')
print('Imports OK')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


## Configuration

In [ ]:
import sys
if '/mnt/e/fyassine/ad-early-detection' not in sys.path:
    sys.path.insert(0, '/mnt/e/fyassine/ad-early-detection')
from DATA.src.splitting.load_splits import splits_dir, split_csv_paths

RANDOM_STATE = 42
set_seed(RANDOM_STATE)   # canonical process-wide seeding from common.seeding

# ── Paths ────────────────────────────────────────────────────────────────
WB_DATA_ROOT  = '/mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__fc_wholebrain_sch200_flat__/matrices'
METADATA_DIR  = '/mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__fc_wholebrain_sch200_flat__/metadata'
SPLITS_DIR    = str(splits_dir('downstream'))
TRAIN_CSV     = os.path.join(SPLITS_DIR, 'train.csv')
VAL_CSV       = os.path.join(SPLITS_DIR, 'val.csv')
TEST_CSV      = os.path.join(SPLITS_DIR, 'test.csv')
ALL_INFO_CSV  = os.path.join(SPLITS_DIR, '_all_splits_patient_info.csv')

# Brain region / atlas parsed from the DATA directory name. Surfaced in the run
# name and stored in the run config so the input data is visible at a glance.
DATA_INFO = region_from_data_root(WB_DATA_ROOT)
REGION    = DATA_INFO['region']
print(f"Input data: region={DATA_INFO['region']}  atlas={DATA_INFO['atlas']}  ({DATA_INFO['dataset_dir']})")

CHECKPOINT_SEARCH_DIRS = [
    str(model_root / 'notebooks' / 'checkpoints' / 'checkpoints_gaae_whole_brain'),
    str(model_root / 'notebooks' / 'checkpoints' / 'checkpoints_gaae_dmn'),
]
OUTPUT_DIR = str(model_root / 'notebooks' / 'checkpoints' / 'checkpoints_gaae_logreg')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── GAAE encoder config (must match checkpoint) ──────────────────────────
CONFIG_PATH = model_root / 'configs' / 'gaae_delcode_whole_brain.json'

# ── This notebook's training hyper-params (loaded from configs/, inline fallback) ─
TRAIN_CONFIG_PATH = model_root / 'configs' / 'gaae_logreg_delcode.json'
_train_defaults = {
    'graph_pool': 'mean', 'lr_C': 1.0, 'lr_max_iter': 2000,
    'lr_class_weight': 'balanced', 'n_folds': 5,
}
if TRAIN_CONFIG_PATH.exists():
    with open(TRAIN_CONFIG_PATH) as _f:
        TRAIN_CONFIG = json.load(_f)
    print(f'Loaded training config from {TRAIN_CONFIG_PATH}')
else:
    TRAIN_CONFIG = _train_defaults
    print(f'Training config not found at {TRAIN_CONFIG_PATH} — using inline defaults.')

# Runner override: merge injected RESOLVED_CONFIG (YAML hyperparams) over JSON config.
if RESOLVED_CONFIG:
    TRAIN_CONFIG = {**TRAIN_CONFIG, **RESOLVED_CONFIG}
    print('Applied RESOLVED_CONFIG overrides from runner.')

GRAPH_POOL      = TRAIN_CONFIG['graph_pool']   # 'mean' | 'max' | 'sum'
LR_C            = TRAIN_CONFIG['lr_C']          # inverse regularisation strength
LR_MAX_ITER     = TRAIN_CONFIG['lr_max_iter']
LR_CLASS_WEIGHT = TRAIN_CONFIG['lr_class_weight']
N_FOLDS         = TRAIN_CONFIG['n_folds']

# Use an existing run dir instead of retraining?
USE_EXISTING_CHECKPOINT = False

print('Config set.')

In [ ]:
# v2 split-hygiene audit — hard-fails if any subject crosses splits.
import sys
from pathlib import Path
_REPO_ROOT = Path('/mnt/e/fyassine/ad-early-detection')
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))
_V2_ROOT = _REPO_ROOT / 'CLASSIFIER'
if str(_V2_ROOT) not in sys.path:
    sys.path.insert(0, str(_V2_ROOT))
from common.sanity import run_full_audit
from DATA.src.splitting.load_splits import split_csv_paths
_ = run_full_audit(split_csv_paths('downstream'))


## Select & load GAAE checkpoint

In [ ]:
from common.checkpoints import select_gaae_checkpoint

GAAE_RUN_NAME, _ckpt_path, GAAE_RUN_DIR = select_gaae_checkpoint(
    CHECKPOINT_SEARCH_DIRS, checkpoint_path=GAAE_CHECKPOINT_PATH)
GAAE_CKPT_PATH = str(_ckpt_path)
print(f'Selected: {GAAE_RUN_NAME}')


In [ ]:
if CONFIG_PATH.exists():
    with open(CONFIG_PATH) as f:
        hp = json.load(f)
    print('Loaded config from', CONFIG_PATH)
else:
    hp = dict(latent_dim=64, num_heads=2, cond_dim=2, dropout=0.3,
              adjacency_k=8, file_variant='z_transformed')
    print('Config not found – using defaults.')

adjacency_args = {'k': hp.get('adjacency_k', 8)}
file_variant   = hp.get('file_variant', 'z_transformed')

_probe = GraphDatasetInMemoryFiltered(
    root=WB_DATA_ROOT,
    adjacency_function=knn_binary_adjacency_matrix_no_diag,
    adjacency_args=adjacency_args,
    filter_csv_path=TEST_CSV,
    file_variant=file_variant,
)
IN_FEATURES = _probe[0].x.size(1)
print(f'IN_FEATURES={IN_FEATURES}  LATENT_DIM={hp.get("latent_dim", 64)}')

from model.GAAE.utils import load_gaae_for_inference
gaae_model = load_gaae_for_inference(
    ckpt_path=Path(GAAE_CKPT_PATH),
    in_features=IN_FEATURES,
    config=hp,
    device=device,
)
print('GAAE encoder loaded and frozen.')


## Load datasets

Merge train+val split CSVs, filter to MCI+converter subjects, build one combined dataset.

In [ ]:
import tempfile

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

# Build merged patient-info CSV (sex/age) once
if not os.path.exists(ALL_INFO_CSV):
    _info = pd.concat(
        [df[['Pseudonym','sex','age']] for df in [train_df, val_df, test_df]],
        ignore_index=True,
    ).drop_duplicates('Pseudonym').reset_index(drop=True)
    _info.to_csv(ALL_INFO_CSV, index=False)
    print(f'Wrote {ALL_INFO_CSV}')

# CV pool = train + val (test held out)
cv_pool_df = pd.concat([train_df, val_df], ignore_index=True)
mci_pool   = cv_pool_df.copy()
test_mci   = test_df.copy()

print('CV pool diagnoses:', mci_pool['diagnosis'].value_counts().to_dict())
print('Test  diagnoses:  ', test_mci['diagnosis'].value_counts().to_dict())

def make_dataset(subset_df, label='subset'):
    tmp = tempfile.NamedTemporaryFile(suffix='.csv', delete=False, mode='w')
    subset_df.to_csv(tmp.name, index=False)
    tmp.close()
    ds = GraphDatasetInMemoryFiltered(
        root=WB_DATA_ROOT,
        adjacency_function=knn_binary_adjacency_matrix_no_diag,
        adjacency_args=adjacency_args,
        filter_csv_path=tmp.name,
        patient_info_path=ALL_INFO_CSV,
        separator=',',
        file_variant=file_variant,
    )
    print(f'{label}: {len(ds)} scans from {subset_df["Pseudonym"].nunique()} subjects')
    return ds

cv_dataset   = make_dataset(mci_pool, 'CV pool')
test_dataset = make_dataset(test_mci,  'Test')


## Embedding extraction

In [ ]:
from model.GAAE.inference import extract_embeddings

cv_emb, cv_pids_raw = extract_embeddings(gaae_model, cv_dataset, device, batch_size=32, graph_pool=GRAPH_POOL)
te_emb, te_pids_raw = extract_embeddings(gaae_model, test_dataset, device, batch_size=32, graph_pool=GRAPH_POOL)

# Strip scan suffix for subject-level CV grouping
cv_pids = [p.split('_')[0] for p in cv_pids_raw]
te_pids = [p.split('_')[0] for p in te_pids_raw]

# Build labels (1=converter, 0=stable-mci)
id2diag = dict(zip(mci_pool['Pseudonym'].astype(str), mci_pool['diagnosis']))
id2diag.update(dict(zip(test_mci['Pseudonym'].astype(str), test_mci['diagnosis'])))

cv_labels  = np.array([1 if id2diag.get(p, 'mci') == 'converter' else 0 for p in cv_pids])
te_labels  = np.array([1 if id2diag.get(p, 'mci') == 'converter' else 0 for p in te_pids])
cv_groups  = np.array(cv_pids)

print(f'CV:   {cv_emb.shape}  converter={cv_labels.sum()}  mci={(1-cv_labels).sum()}')
print(f'Test: {te_emb.shape}  converter={te_labels.sum()}  mci={(1-te_labels).sum()}')


## 5-Fold Stratified Subject-Level Cross-Validation

Each fold trains a `StandardScaler + LogisticRegression` on the GAAE embeddings.
Groups ensure a subject's scans never split across train/val.

In [ ]:
from model.classification.logreg_cv import train_logreg_cv

result = train_logreg_cv(
    X=cv_emb, y=cv_labels, groups=cv_groups,
    n_folds=N_FOLDS, seed=SEED,
    lr_C=LR_C, lr_max_iter=LR_MAX_ITER, lr_class_weight=LR_CLASS_WEIGHT,
)

# Unpack for downstream cells that read these names
best_scaler  = result.best_scaler
best_clf     = result.best_clf
best_val_auc = result.best_val_auc
best_fold    = result.best_fold + 1   # 1-indexed for display
best_threshold_overall = result.youden_threshold
best_f1_threshold      = result.f1_oof_threshold
oof_arr = result.oof_probs
oof_tgt = result.oof_targets
cv_results = {
    'fold':            list(range(1, N_FOLDS + 1)),
    'val_auc':         result.fold_aucs,
    'val_sensitivity': result.fold_sensitivities,
    'val_specificity': result.fold_specificities,
    'val_f1':          result.fold_f1s,
    'best_threshold':  result.fold_thresholds,
}

for fold, auc, sens, spec, f1, thr in zip(
    cv_results['fold'], cv_results['val_auc'], cv_results['val_sensitivity'],
    cv_results['val_specificity'], cv_results['val_f1'], cv_results['best_threshold']
):
    print(f'Fold {fold}: AUC={auc:.3f}  sens={sens:.3f}  spec={spec:.3f}  F1={f1:.3f}  thr={thr:.3f}')

print(f'\nBest fold: {best_fold}  AUC={best_val_auc:.4f}')
print(f'Youden threshold: {best_threshold_overall:.4f}')
print(f'F1-optimal OOF threshold: {best_f1_threshold:.4f}')


## Cross-Validation Results Summary

In [ ]:
print('\nCross-Validation Summary:')
print('=' * 60)
print(f"{'Metric':<20} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print('-' * 60)
for metric in ['val_auc', 'val_sensitivity', 'val_specificity', 'val_f1']:
    v = cv_results[metric]
    print(f'{metric:<20} {np.mean(v):>10.4f} {np.std(v):>10.4f} {np.min(v):>10.4f} {np.max(v):>10.4f}')
print(f'\nBest fold: {best_fold}  AUC={best_val_auc:.4f}')
print(f'Youden thr: {best_threshold_overall:.4f}   F1-OOF thr: {best_f1_threshold:.4f}')


In [ ]:
def _oof_metrics(thr):
    p = (oof_arr >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(oof_tgt, p).ravel() if len(np.unique(oof_tgt))>1 else (0,0,0,0)
    sens = tp/(tp+fn) if (tp+fn)>0 else 0.0
    spec = tn/(tn+fp) if (tn+fp)>0 else 0.0
    f1   = f1_score(oof_tgt, p, zero_division=0)
    return sens, spec, f1

y_s, y_sp, y_f1 = _oof_metrics(best_threshold_overall)
f_s, f_sp, f_f1 = _oof_metrics(best_f1_threshold)
print('OOF threshold options:')
print(f"  [1] Youden   thr={best_threshold_overall:.4f}  sens={y_s:.3f}  spec={y_sp:.3f}  F1={y_f1:.3f}")
print(f"  [2] Best-F1  thr={best_f1_threshold:.4f}  sens={f_s:.3f}  spec={f_sp:.3f}  F1={f_f1:.3f}")

if THRESHOLD_MODE == 'best-f1':
    ACTIVE_THRESHOLD = best_f1_threshold; THRESHOLD_METHOD = 'oof_f1'
elif THRESHOLD_MODE == 'youden':
    ACTIVE_THRESHOLD = best_threshold_overall; THRESHOLD_METHOD = 'oof_youden'
elif THRESHOLD_MODE == 'fixed':
    if FIXED_THRESHOLD is None:
        raise ValueError("THRESHOLD_MODE='fixed' requires FIXED_THRESHOLD")
    ACTIVE_THRESHOLD = float(FIXED_THRESHOLD); THRESHOLD_METHOD = 'fixed'
elif RUN_DIR is not None:
    raise ValueError(
        "THRESHOLD_MODE is required under the experiment runner "
        "(youden | best-f1 | fixed). Set 'threshold_mode:' in experiments.yaml."
    )
else:
    choice = input('Select threshold [1=Youden (default), 2=Best-F1]: ').strip()
    if choice == '2':
        ACTIVE_THRESHOLD = best_f1_threshold
        THRESHOLD_METHOD = 'oof_f1'
    else:
        ACTIVE_THRESHOLD = best_threshold_overall
        THRESHOLD_METHOD = 'oof_youden'
print(f'Using {THRESHOLD_METHOD} threshold: {ACTIVE_THRESHOLD:.4f}')


## Save best model

In [ ]:
import pickle

# Create the run directory (region embedded in the run name) and save artifacts.
from common import tracking
_wb_exp = {'id': EXPERIMENT_ID or 'logreg-static', 'mode': MODE or 'static', 'model': MODEL or 'LogReg', 'dataset': DATASET or REGION, 'seed': SEED, 'wandb': WANDB_ENABLED}
wandb_run = tracking.init_run(_wb_exp, {**(RESOLVED_CONFIG or {}), 'REGION': REGION})

if RUN_DIR:
    run_dir = Path(RUN_DIR); run_dir.mkdir(parents=True, exist_ok=True)
    run_name = RUN_NAME or run_dir.name
else:
    run_name, run_dir = make_run_dir(OUTPUT_DIR, 'gaae_logreg', DATA_INFO)

# The fitted sklearn objects ARE the full state for a LogReg run — reloading
# scaler.pkl + classifier.pkl reproduces predictions exactly.
with open(run_dir / 'scaler.pkl', 'wb') as f:
    pickle.dump(best_scaler, f)
with open(run_dir / 'classifier.pkl', 'wb') as f:
    pickle.dump(best_clf, f)

model_config = {
    'model_type':      'LogisticRegression',
    'graph_pool':      GRAPH_POOL,
    'lr_C':            LR_C,
    'lr_max_iter':     LR_MAX_ITER,
    'lr_class_weight': str(LR_CLASS_WEIGHT),
    'in_features':     IN_FEATURES,
    'gaae_latent':     OUT_FEATURES,
    'gaae_heads':      NUM_HEADS,
    'gaae_cond_dim':   COND_DIM,
    'gaae_dropout':    DROPOUT,
}
dataset_info = {
    **DATA_INFO,
    'train_csv': TRAIN_CSV, 'val_csv': VAL_CSV, 'test_csv': TEST_CSV,
    'n_folds': N_FOLDS,
}

# Snapshot the source that produced this run ("save code") + git commit.
snapshot_source(run_dir, [
    model_root / 'model' / 'GAAE' / 'models.py',
    model_root / 'model' / 'GAAE' / 'dataset.py',
    model_root / 'model' / 'GAAE' / 'utils.py',
])

run_summary = {
    'run_name':         run_name,
    'data_info':        DATA_INFO,
    'dataset_info':     dataset_info,
    'model_config':     model_config,
    'training_config':  TRAIN_CONFIG,
    'gaae_checkpoint':  GAAE_RUN_NAME,
    'gaae_ckpt_path':   GAAE_CKPT_PATH,
    'graph_pool':       GRAPH_POOL,
    'n_folds':          N_FOLDS,
    'best_fold':        int(best_fold),
    'best_val_auc':     float(best_val_auc),
    'active_threshold': float(ACTIVE_THRESHOLD),
    'threshold_method': THRESHOLD_METHOD,
    'youden_threshold': float(best_threshold_overall),
    'f1_threshold':     float(best_f1_threshold),
    'cv_results':       cv_results,
    'lr_C':             LR_C,
    'lr_class_weight':  str(LR_CLASS_WEIGHT),
    'env':              capture_env(),
    'git':              capture_git_provenance(),
    # ── test metrics populated by the test-eval cell below ──
    'test_auc': None,
    'test_sensitivity': None,
    'test_specificity': None,
    'test_f1': None,
}
write_run_summary(run_dir, run_summary)
print(f'Saved to {run_dir}')
LOGREG_RUN_DIR = run_dir   # used by test-eval to back-patch test metrics

try:
    tracking.log_metrics(wandb_run, {'cv_best_val_auc': float(best_val_auc), 'active_threshold': float(ACTIVE_THRESHOLD)})
except NameError:
    pass


## Test-Set Evaluation

Evaluate the best scaler+classifier on the held-out test set.

In [ ]:
X_te_scaled = best_scaler.transform(te_emb)
te_prob      = best_clf.predict_proba(X_te_scaled)[:, 1]
te_pred      = (te_prob >= ACTIVE_THRESHOLD).astype(int)

te_auc  = roc_auc_score(te_labels, te_prob) if len(np.unique(te_labels)) > 1 else 0.0
cm_te   = confusion_matrix(te_labels, te_pred, labels=[0, 1])
tn,fp,fn,tp = cm_te.ravel() if cm_te.size == 4 else (0,0,0,0)
te_sens = tp / (tp + fn + 1e-9)
te_spec = tn / (tn + fp + 1e-9)
te_f1   = f1_score(te_labels, te_pred, zero_division=0)

print(f'Test AUC={te_auc:.4f}  Sens={te_sens:.4f}  Spec={te_spec:.4f}  F1={te_f1:.4f}')
print()
print(classification_report(te_labels, te_pred, target_names=['stable_mci', 'converter']))

# Back-patch run_summary.json with test metrics
patch_run_summary(LOGREG_RUN_DIR, {
    'metrics': {
        'test_auc': float(te_auc), 'test_f1': float(te_f1),
        'test_sensitivity': float(te_sens), 'test_specificity': float(te_spec),
        'threshold': float(ACTIVE_THRESHOLD), 'threshold_method': THRESHOLD_METHOD,
    },
    'test_auc':           float(te_auc),
    'test_sensitivity':   float(te_sens),
    'test_specificity':   float(te_spec),
    'test_f1':            float(te_f1),
    'test_probabilities': te_prob.tolist(),
    'test_labels':        te_labels.tolist(),
})
print(f'Test metrics saved to {LOGREG_RUN_DIR / "run_summary.json"}')

try:
    tracking.log_metrics(wandb_run, {'test_auc': float(te_auc), 'test_f1': float(te_f1), 'test_sensitivity': float(te_sens), 'test_specificity': float(te_spec)})
    tracking.finish_run(wandb_run)
except NameError:
    pass


## ROC Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: per-fold + OOF
ax = axes[0]
sgkf2 = StratifiedGroupKFold(n_splits=N_FOLDS)
for fold, (tr_i, va_i) in enumerate(sgkf2.split(cv_emb, cv_labels, groups=cv_groups)):
    sc2  = StandardScaler().fit(cv_emb[tr_i])
    clf2 = LogisticRegression(C=LR_C, max_iter=LR_MAX_ITER,
                              class_weight=LR_CLASS_WEIGHT, random_state=RANDOM_STATE)
    clf2.fit(sc2.transform(cv_emb[tr_i]), cv_labels[tr_i])
    p2   = clf2.predict_proba(sc2.transform(cv_emb[va_i]))[:, 1]
    fpr2, tpr2, _ = roc_curve(cv_labels[va_i], p2)
    auc2 = roc_auc_score(cv_labels[va_i], p2)
    ax.plot(fpr2, tpr2, alpha=0.5, lw=1, label=f'Fold {fold+1} ({auc2:.3f})')

oof_fpr, oof_tpr, _ = roc_curve(oof_tgt, oof_arr)
oof_auc = roc_auc_score(oof_tgt, oof_arr)
ax.plot(oof_fpr, oof_tpr, 'k-', lw=2, label=f'OOF ({oof_auc:.3f})')
ax.plot([0,1],[0,1],'--',color='gray',lw=1)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('CV Fold ROC curves')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Right: test set
ax2 = axes[1]
te_fpr, te_tpr, _ = roc_curve(te_labels, te_prob)
ax2.plot(te_fpr, te_tpr, 'r-', lw=2, label=f'Test AUC={te_auc:.3f}')
ax2.axvline(1-te_spec, color='blue', linestyle=':', lw=1.5, label=f'Threshold={ACTIVE_THRESHOLD:.3f}')
ax2.plot([0,1],[0,1],'--',color='gray',lw=1)
ax2.set_xlabel('FPR'); ax2.set_ylabel('TPR')
ax2.set_title('Test-Set ROC')
ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

fig.suptitle(f'GAAE LogReg  |  GAAE: {GAAE_RUN_NAME}', fontsize=11)
plt.tight_layout(); plt.show()


## Robustness Evaluation

Tests the classifier stability under input perturbations (feature noise, edge perturbation, conditioning noise).

In [ ]:
from common.robustness import perturb_graph

noise_methods = ['none', 'feature_noise', 'edge_perturbation', 'conditioning_noise']
noise_levels  = [0.0, 0.05, 0.10, 0.20, 0.30]
n_trials      = 5

def perturb_and_embed(sample, method, noise_level):
    """Perturb one graph sample and return its GAAE embedding."""
    d = perturb_graph(sample, method, noise_level, rng=rng)
    d = d.to(device)
    with torch.no_grad():
        z = gaae_model.encode(
            d.x.unsqueeze(0) if d.x.dim() == 1 else d.x,
            d.edge_index, d.edge_attr,
        )
        emb = (z.mean(0) if GRAPH_POOL == 'mean' else
               z.max(0).values if GRAPH_POOL == 'max' else z.sum(0))
    return emb.cpu().numpy()

# baseline: encode all CV samples without noise
baseline_records = []
for i in range(len(cv_dataset)):
    emb    = perturb_and_embed(cv_dataset[i], 'none', 0.0)
    prob_b = best_clf.predict_proba(best_scaler.transform(emb[None]))[0, 1]
    pred_b = int(prob_b >= ACTIVE_THRESHOLD)
    pid    = str(getattr(cv_dataset[i], 'patient_id', i))
    baseline_records.append({
        'DatasetIndex': i, 'PatientID': pid,
        'TrueLabel': int(cv_labels[i]),
        'Probability': float(prob_b),
        'Predicted': pred_b,
        'AbsMargin': abs(prob_b - ACTIVE_THRESHOLD),
    })
baseline_df = pd.DataFrame(baseline_records)
print(f'Baseline CV accuracy: {(baseline_df["Predicted"]==baseline_df["TrueLabel"]).mean():.3f}')


In [ ]:
# Robustness: per-subject stability across noise methods
rob_records = []
for i in range(len(cv_dataset)):
    sample   = cv_dataset[i]
    true_lbl = int(cv_labels[i])
    pid      = str(getattr(sample, 'patient_id', i))
    for method in noise_methods:
        for nl in noise_levels:
            correct_count = 0
            for _ in range(n_trials):
                emb  = perturb_and_embed(sample, method, nl)
                prob = best_clf.predict_proba(best_scaler.transform(emb[None]))[0,1]
                pred = int(prob >= ACTIVE_THRESHOLD)
                correct_count += int(pred == true_lbl)
            rob_records.append({
                'PatientID': pid, 'TrueLabel': true_lbl,
                'Method': method, 'NoiseLevel': nl,
                'StabilityRate': correct_count / n_trials,
            })

rob_df = pd.DataFrame(rob_records)
diag_map = {1: 'converter', 0: 'mci'}
rob_df['SelectionCohort'] = rob_df['TrueLabel'].map(diag_map)

summary = (
    rob_df.groupby(['SelectionCohort','Method','NoiseLevel'])['StabilityRate']
    .mean().reset_index().rename(columns={'StabilityRate':'CohortStabilityRate'})
)
print('Cohort stability summary (higher is better):')
print(summary.to_string(index=False))


## Robustness Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
palette = {'converter': '#F44336', 'mci': '#2196F3'}

for ax, method in zip(axes, ['feature_noise', 'edge_perturbation']):
    sub = summary[summary['Method'] == method]
    for cohort, grp in sub.groupby('SelectionCohort'):
        ax.plot(grp['NoiseLevel'] * 100, grp['CohortStabilityRate'],
                marker='o', label=cohort, color=palette.get(cohort, 'gray'))
    ax.set_title(method.replace('_', ' ').title())
    ax.set_xlabel('Noise Level (%)')
    ax.set_ylabel('Stability Rate')
    ax.set_ylim(0, 1.05)
    ax.legend(title='Cohort'); ax.grid(alpha=0.3)

fig.suptitle('Robustness: LogReg on GAAE Latent Space', fontsize=12)
plt.tight_layout(); plt.show()


## Per-Class Robustness

Top-5 closest-to-threshold converters and non-converters from the CV set.

In [ ]:
top_conv  = baseline_df[baseline_df['TrueLabel']==1].sort_values('AbsMargin').head(5)
top_mci   = baseline_df[baseline_df['TrueLabel']==0].sort_values('AbsMargin').head(5)
print('Top-5 converters (smallest margin):'); print(top_conv.to_string(index=False))
print('\nTop-5 stable MCI (smallest margin):'); print(top_mci.to_string(index=False))

pc_records = []
for group_name, group_df in [('converter', top_conv), ('stable_mci', top_mci)]:
    for _, row in group_df.iterrows():
        sample   = cv_dataset[int(row['DatasetIndex'])]
        true_lbl = int(row['TrueLabel'])
        pid      = str(row['PatientID'])
        for method in noise_methods:
            for nl in noise_levels:
                for trial in range(n_trials):
                    emb  = perturb_and_embed(sample, method, nl)
                    prob = best_clf.predict_proba(best_scaler.transform(emb[None]))[0,1]
                    pred = int(prob >= ACTIVE_THRESHOLD)
                    pc_records.append({
                        'Group': group_name, 'PatientID': pid,
                        'TrueLabel': true_lbl, 'Method': method,
                        'NoiseLevel': nl, 'Trial': trial,
                        'Probability': float(prob),
                        'CorrectPrediction': int(pred == true_lbl),
                    })

pc_df = pd.DataFrame(pc_records)
pc_summary = (
    pc_df.groupby(['Group','PatientID','Method','NoiseLevel'])['CorrectPrediction']
    .mean().reset_index().rename(columns={'CorrectPrediction':'StabilityRate'})
)

# Plot
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
grp_colors = {'converter': '#F44336', 'stable_mci': '#2196F3'}
for row_i, grp_name in enumerate(['converter', 'stable_mci']):
    grp_sub = pc_summary[pc_summary['Group'] == grp_name]
    for col_i, method in enumerate(['feature_noise', 'edge_perturbation']):
        ax = axes[row_i][col_i]
        msub = grp_sub[grp_sub['Method'] == method]
        for pid, pdata in msub.groupby('PatientID'):
            ax.plot(pdata['NoiseLevel']*100, pdata['StabilityRate'],
                    marker='o', alpha=0.7, color=grp_colors[grp_name], lw=1.5)
        ax.set_title(f'{grp_name}  |  {method.replace("_"," ").title()}')
        ax.set_xlabel('Noise (%)'); ax.set_ylabel('Stability')
        ax.set_ylim(-0.05, 1.05); ax.grid(alpha=0.3)
fig.suptitle('Per-patient robustness (top-5 closest to threshold)', fontsize=12)
plt.tight_layout(); plt.show()


## UMAP coloured by prediction vs ground truth

In [ ]:
try:
    import umap
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1,
                        random_state=RANDOM_STATE, metric='euclidean')
    emb_2d = reducer.fit_transform(cv_emb)

    cv_prob_all = best_clf.predict_proba(best_scaler.transform(cv_emb))[:, 1]
    cv_pred_all = (cv_prob_all >= ACTIVE_THRESHOLD).astype(int)
    correct     = (cv_pred_all == cv_labels).astype(int)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    color_diag = ['#F44336' if l==1 else '#2196F3' for l in cv_labels]
    axes[0].scatter(emb_2d[:,0], emb_2d[:,1], c=color_diag, alpha=0.7, s=25,
                    edgecolors='k', linewidths=0.2)
    axes[0].set_title('Ground truth  (red=converter, blue=mci)')

    color_pred = ['#F44336' if p==1 else '#2196F3' for p in cv_pred_all]
    axes[1].scatter(emb_2d[:,0], emb_2d[:,1], c=color_pred, alpha=0.7, s=25,
                    edgecolors='k', linewidths=0.2)
    axes[1].set_title('Predicted label')

    color_corr = ['#4CAF50' if c==1 else '#FF9800' for c in correct]
    axes[2].scatter(emb_2d[:,0], emb_2d[:,1], c=color_corr, alpha=0.7, s=25,
                    edgecolors='k', linewidths=0.2)
    axes[2].set_title('Correct (green) vs Wrong (orange)')

    for ax in axes:
        ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2'); ax.grid(alpha=0.3)
    fig.suptitle(f'UMAP of CV embeddings  |  GAAE: {GAAE_RUN_NAME}', fontsize=11)
    plt.tight_layout(); plt.show()
except ImportError:
    print('umap-learn not installed: pip install umap-learn')
